**Agentic AI - LangGraph**

Problems with GenAI :-
* Reactive
* No memory - not context-aware regarding what it did previously
* Generic advice (can be solved with RAG)
* Can't take actions (can be solved with Tool Augmentation)
* Can't adapt

An AI Agent tackles all of these problems proactively to achieve the desired goal. Generative AI is a building block of Agentic AI.

Agentic AI is a type of AI that can take up a task or goal from a user and then works toward completing it on its own, with minimal human guidance. It plans, takes action, adapts to changes and seeks help only when necessary.

<u>Characteristics</u> :-
* Autonomous - AI system's ablity to make decisions and take actions on its own to achieve a given goal. Proactive. Autonomy in multiple facets (execution, decision making, tool usage). Can be controlled :-
1. Permission scope - Limit what tools/acions agent can perform independently
2. Human-in-the-Loop (HITL) - insert checkpoints where human approval is req before continuing
3. Override cotrols - Allow users to stop, pause or change agent's behavior at any time
4. Guardrails/Policies - Define ard rules orethical boundaries the agent must follow
* Goal Oriented - AI system operates with persistent objective in mind and continously directs its actions to achieve that goal rather than responding to isolated prompts. Act as compass for autonomy. Can come with constraints. Stored in core memory. Can be altered.
* Planning - Ability to break down a high-level goal into structured seq of actions/subgoals and decide best path to achieve desired outcome. Steps:-
1. Generating multiple candidate plans
2. Evaluate each plan (Efficiency, Tool Availability, Cost, Risk, Alignment with Constraints)
3. Select best plan with help of HITL and a pre-programmed policy
* Reasoning - Cognitive process through with Agentic AI system interprets info, draws conclusions and makes decisions - both planning ahead and executing actions in real time.
1. During planning - Goal decomposition (abstrac goals into concrete steps), tool selection, resource estimation (time, dependencies, risks)
2. During execution - Decision-making, HITL handling (knowing when to pause/ask for help), Error handling
* Adaptability - ability to modify plans, strategies or actions in response to unexpected conditions - while staying aligned with goal. Reasons - Failures, External feedback, Changing goals
* Context Awareness - ability to understand, retain and utilize relevant info from ongoing task, past interactions, user preferences and environmental cues to make better decisions throughout a multi-step process. Implemented in memory (short and long terms) Types of context :-
1. Original Goal
2. Progress till now + Interaction history
3. Environment State
4. Tool responses
5. User specific preferences
6. Policy or Guardrails

<u>Components</u> :-
* Brain - Usually LLM. Tasks - Goal Interpretation, Planning, Reasoning, Tool Selection, Communication
* Orchestrator - Task Sequencing, Conditional Routing, Retry Logic,Looping & Iteration, Delegation
* Tools - External Actions, Knowledge Base Access
* Memory - Short-Term Memory, Long-Term Memory, State Tracking
* Supervisor - Approval Requests (HITL), Guardrails Enforcment, Edge Case Escalation

<u>Problems in LangChain</u> :-

1. Control Flow Complexity - Conditional Branch, Loop, Jump (graphs are non-linear => easier to program complex graphs in LangGraph)
2. Handling State - By default LangChain is stateless, no intrinsic state (LangGraph is stateful)
3. Event Driven Execution - In LangChain, chains are always sequential => not pausable to wait for triggers (can use store(checkpointer) to save state in LangGraph)
4. Fault Tolerance - Not present in LangChain (Builtin in LangGraph - retry logic (small faults), recovery - checkpointer (big faults))
5. HITL - No pause for human intervention for long-running flows (explicitly present to pause indefinitely in LangGraph)
6. Nested Workflows - one node can be replaced by subgraphs in LangGraph (encapsulation) - to make multiagent systems, brings reusability
7. Observability - how easily you can monitor, debug and understand what your workflow is doing at runtime (LangSmith can only monitor LangChain/LangSmith, but not glue code - meaning more glue code => partial observability in LangChain vs complete in LangGraph)

<b><u>LangGraph</b></u> is an orchestration framework to build stateful, multi-step and event-driven workflows using LLMs. Ideal for both Single and Multi Agent Agentic AI apps. A flowchart engine for LLMs - you define steps (nodes), how they're connected (edges) and the logic that governs the transitions. LangGraph takes care of state management, conditional branching, looping, pausing/resuming and fault recovery - essential for robust, production-grade AI systems. It models logic as graph of nodes (tasks) and edges (routing) instead of linear chain.

* LangChain - simple, linear workflows like prompt chain, summarizer or basic retreival system
* LangGraph - complex, non-linear workflows that need - conditional paths, loops, HITL, multi-agent coordination, asynchronous or event-driven execution

LLM Workflow - step by step process to build complex LLM apps. Each step performs a distinct task and can be linear, parallel, branched or looped. Common workflows:-
1. Prompt Chaining (sequential)
2. Routing (branch)
3. Parallelization (parallel)
4. Orchestrator Workers (not predefined parallelization)
5. Evaluator Optimizer (rejected+feedback)

State - shared memory that flows through workflow - holds all data passed b/w nodes as the graph runs. Every node has access to this state, is of TypedDict class and it is mutable .

Reducers - define how updates from nodes are applied to shared state. Each key in state can have its own reducer, which determines if new data replaces, merges or adds to existing value.

LangGraph Execution Model :-
1. Graph definition - state schema, nodes (tasks), edges (which nodes are connected)
2. Compilation - .compile on stategraph - checks graph structure
3. Invocation - run graph with .invoke(initial_state). LangGraph sends initial stage as message to entry node(s) (message passing).
4. Super-Steps begin - Execution proceeds in rounds (super-steps - all steps in one level of hierarchy)
5. Message passing & node activation - Messages are passed to downstream nodes via edges. Nodes that receive messages become active for next round.
6. Halting condition - Execution stops when no nodes are active and no messages are in transit.

Persistence - Ability to save and restore state of a workflow over time

Benefits:-
1. Short Term memory
2. Fault Tolerance
3. HITL
4. Time Travel

Streaming - model starts sending tokens as soon as they're generated, instead of waiting for the entire response to be ready before returning it

Benefits :-
1. Faster response time - low drop-off rates
2. Mimics human-like conversation (feels alive, keeps user engaged, builds trust)
3. Important for multi-modal UIs
4. Better UX for long o/p such as code
5. Can cancel midway saving tokens
6. Can interleave UI updates eg show "thinking", show tool results

ToolNode - prebuilt node type that acts as a bridge b/w graph and external tools -> listedn for tool_calls from LLM (like call_search() or get_weather()) and automatically route the request to correct tool, then pass the o/p to the graph.

Tools condition - prebuilt conditional edge condition that helps graph decide if the flow should go to ToolNode next or back to the LLM/end.